<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
import os
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma4:e4b", backend="openai", api_base="http://localhost:11434/v1/")

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q8_0", backend="litellm", api_base="http://localhost:8080")

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-E4B-it-GGUF", backend="litellm", api_base="http://localhost:1234/v1")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Adding **`Memory`**

In the previous chapter, we covered which LLM to choose using various inference engines. In this chapter we will cover how to give it memory:

![../images/ch4.png](../images/ch4.png)

The issue with the `TinyAgent` that we have thus far, is that it does not track and remember its previous conversations, it is stateless. Let us demonstrate with an example by using the Agent from Chapter 2:

In [2]:
from illustrated_agents.chapters.ch2 import TinyAgent

ch_2_agent = TinyAgent(llm=llm)
response = ch_2_agent.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Hi Maarten and Jay! It's a pleasure to meet you.

"An Illustrated Guide to AI Agents" sounds like a fantastic and timely resource, especially given how rapidly the field of AI agents is evolving.

How can I help you today? Are you looking for feedback on your book, wanting to discuss a specific concept related to your content, or something else entirely?


When we query the model again asking whether it knows our names, it seems to have forgotten them! In fact, it actually hasn't forgotten our name but instead has never received. Everytime you query an LLM it starts from an blank slate, one you have to fill yourself. So without telling the model the our conversation history, it has no way of knowing.

In [3]:
response = ch_2_agent.run("Hi! What are our names?")
print(response)

My name is Gemma 4. I am a Large Language Model, and I was developed by Google DeepMind.

I don't have access to a record of our previous interactions, so I don't know what your name is! You'll have to tell me.


## 3 - The **`Memory`** Module

As covered in the book, there are many ways to build up memory which can be quite difficult. In this example, we are going to keep it simple and only track the conversation history.

The `Memory` that we are going to build uses the `messages` structure for tracking conversations:

```json
[
    {
        "role": "user",
        "content": "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."
    },
    {
        "role": "assistant",
        "content": "Hi Maarten and Jay! It's wonderful to meet you."
    }
]
```

This module is rather straightforward and appends new messages each time the user makes a query or when the LLM gives back a reply. As such, the `Memory` module only requires a few lines of code:

In [4]:
from illustrated_agents.chapters.ch2 import Response

class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str):
        """Add a message to memory."""
        self.messages.append({"role": role, "content": content})

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

We can annotate this module and explore each function in more detail:

In [5]:
from illustrated_agents.chapters.ch4 import memory_annotated; memory_annotated

## 4 - Updating `agent.py`

This added to the `TinyAgent`, which also requires updating a `_step` to track the conversation history following three steps:

1. The user's query is added to the `Memory` module
2. Based on the current memory, the LLM generates a response.
3. The response of the LLM is added to the `Memory` module.

In [6]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory):
        self.llm = llm
        self.memory = memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning
        self.reflector = None  # Chapter 6: Add Reflection
        self.skills = None  # Chapter 6: Add Skills

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)

        return self._step()

    def _step(self) -> str:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response.content)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        # Placeholder - will be implemented in later chapters
        return f"Executed action: {action}"

Let's highlight these steps:

In [7]:
from illustrated_agents.chapters.ch4 import tinyagent_annotated; tinyagent_annotated


Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [8]:
from illustrated_agents.chapters.ch4 import tinyagents_diff; tinyagents_diff

Next, let's create our `TinyAgent` with `Memory`:

In [9]:
# Add memory to the Agent
memory = Memory()
agent_with_memory = TinyAgent(llm=llm, memory=memory)

We can start filling up the memory by conversing with the model as we did before:

In [10]:
response = agent_with_memory.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Hi Maarten and Jay! It's a pleasure to meet you both. "An Illustrated Guide to AI Agents" sounds like a really valuable and engaging resource.

How can I help you today? Are you looking for feedback on the book, ideas for promotion, summarizing concepts, or something else entirely? Let me know what you have in mind! 😊


Now that we have memory, we can ask a follow-up question to the original conversation and see if it remember our names correctly.

In [11]:
response = agent_with_memory.run("Hi! What are our names?")
print(response)

Your names are **Maarten and Jay**.


It does! The method by which it does so is filling up the conversation history. Let's see what the current state is.

In [12]:
agent_with_memory.memory.get_messages()

[{'role': 'user',
  'content': "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."},
 {'role': 'assistant',
  'content': 'Hi Maarten and Jay! It\'s a pleasure to meet you both. "An Illustrated Guide to AI Agents" sounds like a really valuable and engaging resource.\n\nHow can I help you today? Are you looking for feedback on the book, ideas for promotion, summarizing concepts, or something else entirely? Let me know what you have in mind! 😊'},
 {'role': 'user', 'content': 'Hi! What are our names?'},
 {'role': 'assistant', 'content': 'Your names are **Maarten and Jay**.'}]

Note that this entire list is given to the LLM whenever we ask it a new question. That way, it "remembers" the conversation we had before. We say "remembers" because even though it may look like it, it actually has no internal memory. We merely tell the model what the conversation was!

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [13]:
from illustrated_agents.chapters.ch4 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py  ← Updated (Integrated `Memory` into your `TinyAgent`.)                                            │
│ ├── llm.py                                                                                                      │
│ └── memory.py ← New (Track conversation history.)                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# ▂▂▂▂▂▂▂▂▂▂▂▂